# Autoresearch-Trader Experiment Analysis

Analysis of autonomous trading strategy experiments from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
for col in ["sharpe_ratio", "total_return", "max_drawdown", "avg_turnover",
            "training_seconds", "total_seconds", "peak_vram_mb", "num_params"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["is_record"] = df["sharpe_ratio"].cummax() == df["sharpe_ratio"]
# First occurrence of each cummax value only
df["is_record"] = df["is_record"] & (~df["sharpe_ratio"].duplicated(keep="first"))

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df

In [ ]:
n_positive = (df["sharpe_ratio"] > 0).sum()
n_negative = (df["sharpe_ratio"] <= 0).sum()
n_records = df["is_record"].sum()

print(f"Positive Sharpe: {n_positive}/{len(df)}")
print(f"Negative Sharpe: {n_negative}/{len(df)}")
print(f"New records set:  {n_records}")
print(f"Hit rate (positive): {n_positive / len(df):.1%}")

In [ ]:
# Show record-setting experiments (each one was the best at its time)
records = df[df["is_record"]].copy()
print(f"Record-setting experiments ({len(records)} total):\n")
for i, row in records.iterrows():
    sr = row["sharpe_ratio"]
    tr = row["total_return"]
    dd = row["max_drawdown"]
    desc = row["description"]
    print(f"  #{i:3d}  sharpe={sr:+.4f}  ret={tr:+.4f}  dd={dd:.4f}  {desc}")

## Sharpe Ratio Over Time

Track how Sharpe ratio evolves across experiments. The running maximum shows the frontier — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

baseline_sr = df.iloc[0]["sharpe_ratio"]
best_sr = df["sharpe_ratio"].max()

# Non-record experiments as faint dots
non_rec = df[~df["is_record"]]
ax.scatter(non_rec.index, non_rec["sharpe_ratio"],
           c="#cccccc", s=15, alpha=0.6, zorder=2, label="Experiments")

# Record-setters as prominent dots
rec = df[df["is_record"]]
ax.scatter(rec.index, rec["sharpe_ratio"],
           c="#2ecc71", s=60, zorder=4, label="New record",
           edgecolors="black", linewidths=0.5)

# Running maximum step line
running_max = df["sharpe_ratio"].cummax()
ax.step(df.index, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Zero line
ax.axhline(y=0, color="#e74c3c", linestyle="--", alpha=0.4, linewidth=1, label="Sharpe = 0")

# Label each record with its tag + description
for idx in rec.index:
    sr = df.loc[idx, "sharpe_ratio"]
    tag = str(df.loc[idx, "tag"]).strip()
    desc = str(df.loc[idx, "description"]).strip()
    label = f"{tag}: {desc}"
    if len(label) > 50:
        label = label[:47] + "..."
    ax.annotate(label, (idx, sr),
                textcoords="offset points",
                xytext=(6, 8), fontsize=7.5,
                color="#1a7a3a", alpha=0.9,
                rotation=25, ha="left", va="bottom")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Sharpe Ratio (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch-Trader Progress: {len(df)} Experiments, Best Sharpe = {best_sr:.4f}", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Risk-Return Landscape

Sharpe vs max drawdown across experiments. The ideal is top-left: high Sharpe, shallow drawdown.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

valid = df.dropna(subset=["sharpe_ratio", "max_drawdown"]).copy()
colors = ["#2ecc71" if r else "#95a5a6" for r in valid["is_record"]]
sizes = [80 if r else 25 for r in valid["is_record"]]

ax.scatter(valid["max_drawdown"] * 100, valid["sharpe_ratio"],
           c=colors, s=sizes, alpha=0.7, edgecolors="black", linewidths=0.3)

for idx in valid[valid["is_record"]].index:
    tag = str(valid.loc[idx, "tag"]).strip()
    ax.annotate(tag,
                (valid.loc[idx, "max_drawdown"] * 100, valid.loc[idx, "sharpe_ratio"]),
                textcoords="offset points", xytext=(6, 4),
                fontsize=8, color="#1a7a3a", alpha=0.9)

ax.axhline(y=0, color="#e74c3c", linestyle="--", alpha=0.3)
ax.set_xlabel("Max Drawdown (%)", fontsize=12)
ax.set_ylabel("Sharpe Ratio", fontsize=12)
ax.set_title("Risk-Return Landscape", fontsize=14)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("risk_return.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary Statistics

In [ ]:
baseline_sr = df.iloc[0]["sharpe_ratio"]
best_row = df.loc[df["sharpe_ratio"].idxmax()]
best_sr = best_row["sharpe_ratio"]

print(f"Baseline Sharpe:   {baseline_sr:+.4f}")
print(f"Best Sharpe:       {best_sr:+.4f}")
print(f"Improvement:       {best_sr - baseline_sr:+.4f}")
print(f"Best experiment:   {best_row['tag']} — {best_row['description']}")
print(f"Best total return: {best_row['total_return']:+.4f} ({best_row['total_return']*100:+.2f}%)")
print(f"Best max drawdown: {best_row['max_drawdown']:.4f} ({best_row['max_drawdown']*100:.2f}%)")
print()

print("Record-setting progression:")
records = df[df["is_record"]].copy()
for i, (_, row) in enumerate(records.iterrows()):
    tag = str(row["tag"]).strip()
    print(f"  {tag:20s}  sharpe={row['sharpe_ratio']:+.4f}  ret={row['total_return']:+.4f}  dd={row['max_drawdown']:.4f}")

## Top Hits (Ranked by Sharpe Improvement)

In [ ]:
# Each record's delta vs the previous record
records = df[df["is_record"]].copy()
records["prev_sharpe"] = records["sharpe_ratio"].shift(1)
records["delta"] = records["sharpe_ratio"] - records["prev_sharpe"]

hits = records.iloc[1:].copy()  # drop baseline
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Sharpe':>8}  {'Tag':<20s}  Description")
print("-" * 100)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    tag = str(row["tag"]).strip()
    desc = str(row["description"]).strip()
    print(f"{rank:4d}  {row['delta']:+.4f}  {row['sharpe_ratio']:+.4f}  {tag:<20s}  {desc}")

print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>8}  {'TOTAL':<20s}  improvement over baseline")

## Turnover vs Sharpe

Higher turnover means more transaction costs. The best strategies find signal without excessive trading.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

valid = df.dropna(subset=["sharpe_ratio", "avg_turnover"]).copy()
colors = ["#2ecc71" if r else "#95a5a6" for r in valid["is_record"]]
sizes = [80 if r else 25 for r in valid["is_record"]]

ax.scatter(valid["avg_turnover"], valid["sharpe_ratio"],
           c=colors, s=sizes, alpha=0.7, edgecolors="black", linewidths=0.3)

for idx in valid[valid["is_record"]].index:
    tag = str(valid.loc[idx, "tag"]).strip()
    ax.annotate(tag,
                (valid.loc[idx, "avg_turnover"], valid.loc[idx, "sharpe_ratio"]),
                textcoords="offset points", xytext=(6, 4),
                fontsize=8, color="#1a7a3a", alpha=0.9)

ax.axhline(y=0, color="#e74c3c", linestyle="--", alpha=0.3)
ax.set_xlabel("Average Daily Turnover", fontsize=12)
ax.set_ylabel("Sharpe Ratio", fontsize=12)
ax.set_title("Turnover vs Performance", fontsize=14)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("turnover_sharpe.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Full results table, sorted by Sharpe
display_cols = ["tag", "sharpe_ratio", "total_return", "max_drawdown",
                "avg_turnover", "total_seconds", "num_params", "description"]
cols = [c for c in display_cols if c in df.columns]
print(df[cols].sort_values("sharpe_ratio", ascending=False).to_string(index=False))